In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

import numpy as np
import MDAnalysis as mda
import os
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score , confusion_matrix , brier_score_loss , log_loss , f1_score
from sklearn.calibration import calibration_curve

from sklearn.preprocessing import normalize

In [ ]:
model_scaler_dir = os.path.join("../" , "heavy_data_NO_GIT/models_scalers/")
data_array_dir = os.path.join("../" , "heavy_data_NO_GIT/data_arrays/")

In [ ]:
prob_threshold = 0.5

In [ ]:
def ece(y_true, y_prob, n_bins=10):
    """
    Expected Calibration Error for a binary classifier.

    Bins predictions by confidence, then measures how far the observed
    fraction of positives (accuracy) sits from the mean predicted
    probability (confidence) in each bin, weighted by bin size.
    Empty bins are skipped.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    # index of the bin each prediction falls into (0 .. n_bins-1)
    bin_ids = np.clip(np.digitize(y_prob, edges[1:-1], right=False), 0, n_bins - 1)

    total = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if not mask.any():                 # skip empty bins
            continue
        acc  = y_true[mask].mean()         # fraction of positives in this bin
        conf = y_prob[mask].mean()         # mean predicted probability in this bin
        total += (mask.sum() / len(y_true)) * abs(acc - conf)

    return total



def min_max_normalize(arr):
    arr = np.asarray(arr, dtype=float) # Convert to float array
    return (arr - arr.min()) / (arr.max() - arr.min())

## RAW

In [ ]:
test_sn1_raw_path = os.path.join(data_array_dir , "nn_test_sn1_raw.pkl")

with open(test_sn1_raw_path , 'rb') as f:
    X_sn1_raw , y_sn1_raw = pickle.load(f)

model_sn1_raw_path = os.path.join(model_scaler_dir , "nn_sn1_raw_model_ft_last.sav")
scaler_sn1_raw_path = os.path.join(model_scaler_dir , "nn_sn1_raw_scaler_ft_last.sav")

model_sn1_raw = pickle.load(open(model_sn1_raw_path, 'rb'))
scaler_sn1_raw = pickle.load(open(scaler_sn1_raw_path, 'rb'))

X_sn1_raw_test_scaled = scaler_sn1_raw.transform(X_sn1_raw)
nn_sn1_raw_pred_prob = model_sn1_raw.predict(X_sn1_raw_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn1_raw_pred = (nn_sn1_raw_pred_prob > prob_threshold).astype(int)

test_sn2_raw_path = os.path.join(data_array_dir , "nn_test_sn2_raw.pkl")

with open(test_sn2_raw_path , 'rb') as f:
    X_sn2_raw , y_sn2_raw = pickle.load(f)

model_sn2_raw_path = os.path.join(model_scaler_dir , "nn_sn2_raw_model_ft_last.sav")
scaler_sn2_raw_path = os.path.join(model_scaler_dir , "nn_sn2_raw_scaler_ft_last.sav")

model_sn2_raw = pickle.load(open(model_sn2_raw_path, 'rb'))
scaler_sn2_raw = pickle.load(open(scaler_sn2_raw_path, 'rb'))

X_sn2_raw_test_scaled = scaler_sn2_raw.transform(X_sn2_raw)
nn_sn2_raw_pred_prob = model_sn2_raw.predict(X_sn2_raw_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn2_raw_pred = (nn_sn2_raw_pred_prob > prob_threshold).astype(int)

nn_acc_sn1_raw = accuracy_score(y_sn1_raw, nn_sn1_raw_pred)
nn_acc_sn2_raw = accuracy_score(y_sn2_raw, nn_sn2_raw_pred)

## 3 pt

In [ ]:
test_sn1_3pt_path = os.path.join(data_array_dir , "nn_test_sn1_3pt.pkl")

with open(test_sn1_3pt_path , 'rb') as f:
    X_sn1_3pt , y_sn1_3pt = pickle.load(f)

model_sn1_3pt_path = os.path.join(model_scaler_dir , "nn_sn1_3pt_model_ft_last.sav")
scaler_sn1_3pt_path = os.path.join(model_scaler_dir , "nn_sn1_3pt_scaler_ft_last.sav")

model_sn1_3pt = pickle.load(open(model_sn1_3pt_path, 'rb'))
scaler_sn1_3pt = pickle.load(open(scaler_sn1_3pt_path, 'rb'))

X_sn1_3pt_test_scaled = scaler_sn1_3pt.transform(X_sn1_3pt)
nn_sn1_3pt_pred_prob = model_sn1_3pt.predict(X_sn1_3pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn1_3pt_pred = (nn_sn1_3pt_pred_prob > prob_threshold).astype(int)

test_sn2_3pt_path = os.path.join(data_array_dir , "nn_test_sn2_3pt.pkl")

with open(test_sn2_3pt_path , 'rb') as f:
    X_sn2_3pt , y_sn2_3pt = pickle.load(f)

model_sn2_3pt_path = os.path.join(model_scaler_dir , "nn_sn2_3pt_model_ft_last.sav")
scaler_sn2_3pt_path = os.path.join(model_scaler_dir , "nn_sn2_3pt_scaler_ft_last.sav")

model_sn2_3pt = pickle.load(open(model_sn2_3pt_path, 'rb'))
scaler_sn2_3pt = pickle.load(open(scaler_sn2_3pt_path, 'rb'))

X_sn2_3pt_test_scaled = scaler_sn2_3pt.transform(X_sn2_3pt)
nn_sn2_3pt_pred_prob = model_sn2_3pt.predict(X_sn2_3pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn2_3pt_pred = (nn_sn2_3pt_pred_prob > prob_threshold).astype(int)

nn_acc_sn1_3pt = accuracy_score(y_sn1_3pt, nn_sn1_3pt_pred)
nn_acc_sn2_3pt = accuracy_score(y_sn2_3pt, nn_sn2_3pt_pred)

## 5 pt

In [ ]:
test_sn1_5pt_path = os.path.join(data_array_dir , "nn_test_sn1_5pt.pkl")

with open(test_sn1_5pt_path , 'rb') as f:
    X_sn1_5pt , y_sn1_5pt = pickle.load(f)

model_sn1_5pt_path = os.path.join(model_scaler_dir , "nn_sn1_5pt_model_ft_last.sav")
scaler_sn1_5pt_path = os.path.join(model_scaler_dir , "nn_sn1_5pt_scaler_ft_last.sav")

model_sn1_5pt = pickle.load(open(model_sn1_5pt_path, 'rb'))
scaler_sn1_5pt = pickle.load(open(scaler_sn1_5pt_path, 'rb'))

X_sn1_5pt_test_scaled = scaler_sn1_5pt.transform(X_sn1_5pt)
nn_sn1_5pt_pred_prob = model_sn1_5pt.predict(X_sn1_5pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn1_5pt_pred = (nn_sn1_5pt_pred_prob > prob_threshold).astype(int)

test_sn2_5pt_path = os.path.join(data_array_dir , "nn_test_sn2_5pt.pkl")

with open(test_sn2_5pt_path , 'rb') as f:
    X_sn2_5pt , y_sn2_5pt = pickle.load(f)

model_sn2_5pt_path = os.path.join(model_scaler_dir , "nn_sn2_5pt_model_ft_last.sav")
scaler_sn2_5pt_path = os.path.join(model_scaler_dir , "nn_sn2_5pt_scaler_ft_last.sav")

model_sn2_5pt = pickle.load(open(model_sn2_5pt_path, 'rb'))
scaler_sn2_5pt = pickle.load(open(scaler_sn2_5pt_path, 'rb'))

X_sn2_5pt_test_scaled = scaler_sn2_5pt.transform(X_sn2_5pt)
nn_sn2_5pt_pred_prob = model_sn2_5pt.predict(X_sn2_5pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn2_5pt_pred = (nn_sn2_5pt_pred_prob > prob_threshold).astype(int)

nn_acc_sn1_5pt = accuracy_score(y_sn1_5pt, nn_sn1_5pt_pred)
nn_acc_sn2_5pt = accuracy_score(y_sn2_5pt, nn_sn2_5pt_pred)

## 5 pt

In [ ]:
test_sn1_5pt_path = os.path.join(data_array_dir , "nn_test_sn1_5pt.pkl")

with open(test_sn1_5pt_path , 'rb') as f:
    X_sn1_5pt , y_sn1_5pt = pickle.load(f)

model_sn1_5pt_path = os.path.join(model_scaler_dir , "nn_sn1_5pt_model_ft_last.sav")
scaler_sn1_5pt_path = os.path.join(model_scaler_dir , "nn_sn1_5pt_scaler_ft_last.sav")

model_sn1_5pt = pickle.load(open(model_sn1_5pt_path, 'rb'))
scaler_sn1_5pt = pickle.load(open(scaler_sn1_5pt_path, 'rb'))

X_sn1_5pt_test_scaled = scaler_sn1_5pt.transform(X_sn1_5pt)
nn_sn1_5pt_pred_prob = model_sn1_5pt.predict(X_sn1_5pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn1_5pt_pred = (nn_sn1_5pt_pred_prob > prob_threshold).astype(int)

test_sn2_5pt_path = os.path.join(data_array_dir , "nn_test_sn2_5pt.pkl")

with open(test_sn2_5pt_path , 'rb') as f:
    X_sn2_5pt , y_sn2_5pt = pickle.load(f)

model_sn2_5pt_path = os.path.join(model_scaler_dir , "nn_sn2_5pt_model_ft_last.sav")
scaler_sn2_5pt_path = os.path.join(model_scaler_dir , "nn_sn2_5pt_scaler_ft_last.sav")

model_sn2_5pt = pickle.load(open(model_sn2_5pt_path, 'rb'))
scaler_sn2_5pt = pickle.load(open(scaler_sn2_5pt_path, 'rb'))

X_sn2_5pt_test_scaled = scaler_sn2_5pt.transform(X_sn2_5pt)
nn_sn2_5pt_pred_prob = model_sn2_5pt.predict(X_sn2_5pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn2_5pt_pred = (nn_sn2_5pt_pred_prob > prob_threshold).astype(int)

nn_acc_sn1_5pt = accuracy_score(y_sn1_5pt, nn_sn1_5pt_pred)
nn_acc_sn2_5pt = accuracy_score(y_sn2_5pt, nn_sn2_5pt_pred)

## 9 pt

In [ ]:
test_sn1_9pt_path = os.path.join(data_array_dir , "nn_test_sn1_9pt.pkl")

with open(test_sn1_9pt_path , 'rb') as f:
    X_sn1_9pt , y_sn1_9pt = pickle.load(f)

model_sn1_9pt_path = os.path.join(model_scaler_dir , "nn_sn1_9pt_model_ft_last.sav")
scaler_sn1_9pt_path = os.path.join(model_scaler_dir , "nn_sn1_9pt_scaler_ft_last.sav")

model_sn1_9pt = pickle.load(open(model_sn1_9pt_path, 'rb'))
scaler_sn1_9pt = pickle.load(open(scaler_sn1_9pt_path, 'rb'))

X_sn1_9pt_test_scaled = scaler_sn1_9pt.transform(X_sn1_9pt)
nn_sn1_9pt_pred_prob = model_sn1_9pt.predict(X_sn1_9pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn1_9pt_pred = (nn_sn1_9pt_pred_prob > prob_threshold).astype(int)

test_sn2_9pt_path = os.path.join(data_array_dir , "nn_test_sn2_9pt.pkl")

with open(test_sn2_9pt_path , 'rb') as f:
    X_sn2_9pt , y_sn2_9pt = pickle.load(f)

model_sn2_9pt_path = os.path.join(model_scaler_dir , "nn_sn2_9pt_model_ft_last.sav")
scaler_sn2_9pt_path = os.path.join(model_scaler_dir , "nn_sn2_9pt_scaler_ft_last.sav")

model_sn2_9pt = pickle.load(open(model_sn2_9pt_path, 'rb'))
scaler_sn2_9pt = pickle.load(open(scaler_sn2_9pt_path, 'rb'))

X_sn2_9pt_test_scaled = scaler_sn2_9pt.transform(X_sn2_9pt)
nn_sn2_9pt_pred_prob = model_sn2_9pt.predict(X_sn2_9pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn2_9pt_pred = (nn_sn2_9pt_pred_prob > prob_threshold).astype(int)

nn_acc_sn1_9pt = accuracy_score(y_sn1_9pt, nn_sn1_9pt_pred)
nn_acc_sn2_9pt = accuracy_score(y_sn2_9pt, nn_sn2_9pt_pred)

## 11 pt

In [ ]:
test_sn1_11pt_path = os.path.join(data_array_dir , "nn_test_sn1_11pt.pkl")

with open(test_sn1_11pt_path , 'rb') as f:
    X_sn1_11pt , y_sn1_11pt = pickle.load(f)

model_sn1_11pt_path = os.path.join(model_scaler_dir , "nn_sn1_11pt_model_ft_last.sav")
scaler_sn1_11pt_path = os.path.join(model_scaler_dir , "nn_sn1_11pt_scaler_ft_last.sav")

model_sn1_11pt = pickle.load(open(model_sn1_11pt_path, 'rb'))
scaler_sn1_11pt = pickle.load(open(scaler_sn1_11pt_path, 'rb'))

X_sn1_11pt_test_scaled = scaler_sn1_11pt.transform(X_sn1_11pt)
nn_sn1_11pt_pred_prob = model_sn1_11pt.predict(X_sn1_11pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn1_11pt_pred = (nn_sn1_11pt_pred_prob > prob_threshold).astype(int)

test_sn2_11pt_path = os.path.join(data_array_dir , "nn_test_sn2_11pt.pkl")

with open(test_sn2_11pt_path , 'rb') as f:
    X_sn2_11pt , y_sn2_11pt = pickle.load(f)

model_sn2_11pt_path = os.path.join(model_scaler_dir , "nn_sn2_11pt_model_ft_last.sav")
scaler_sn2_11pt_path = os.path.join(model_scaler_dir , "nn_sn2_11pt_scaler_ft_last.sav")

model_sn2_11pt = pickle.load(open(model_sn2_11pt_path, 'rb'))
scaler_sn2_11pt = pickle.load(open(scaler_sn2_11pt_path, 'rb'))

X_sn2_11pt_test_scaled = scaler_sn2_11pt.transform(X_sn2_11pt)
nn_sn2_11pt_pred_prob = model_sn2_11pt.predict(X_sn2_11pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn2_11pt_pred = (nn_sn2_11pt_pred_prob > prob_threshold).astype(int)

nn_acc_sn1_11pt = accuracy_score(y_sn1_11pt, nn_sn1_11pt_pred)
nn_acc_sn2_11pt = accuracy_score(y_sn2_11pt, nn_sn2_11pt_pred)

## 13 pt

In [ ]:
test_sn1_13pt_path = os.path.join(data_array_dir , "nn_test_sn1_13pt.pkl")

with open(test_sn1_13pt_path , 'rb') as f:
    X_sn1_13pt , y_sn1_13pt = pickle.load(f)

model_sn1_13pt_path = os.path.join(model_scaler_dir , "nn_sn1_13pt_model_ft_last.sav")
scaler_sn1_13pt_path = os.path.join(model_scaler_dir , "nn_sn1_13pt_scaler_ft_last.sav")

model_sn1_13pt = pickle.load(open(model_sn1_13pt_path, 'rb'))
scaler_sn1_13pt = pickle.load(open(scaler_sn1_13pt_path, 'rb'))

X_sn1_13pt_test_scaled = scaler_sn1_13pt.transform(X_sn1_13pt)
nn_sn1_13pt_pred_prob = model_sn1_13pt.predict(X_sn1_13pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn1_13pt_pred = (nn_sn1_13pt_pred_prob > prob_threshold).astype(int)

test_sn2_13pt_path = os.path.join(data_array_dir , "nn_test_sn2_13pt.pkl")

with open(test_sn2_13pt_path , 'rb') as f:
    X_sn2_13pt , y_sn2_13pt = pickle.load(f)

model_sn2_13pt_path = os.path.join(model_scaler_dir , "nn_sn2_13pt_model_ft_last.sav")
scaler_sn2_13pt_path = os.path.join(model_scaler_dir , "nn_sn2_13pt_scaler_ft_last.sav")

model_sn2_13pt = pickle.load(open(model_sn2_13pt_path, 'rb'))
scaler_sn2_13pt = pickle.load(open(scaler_sn2_13pt_path, 'rb'))

X_sn2_13pt_test_scaled = scaler_sn2_13pt.transform(X_sn2_13pt)
nn_sn2_13pt_pred_prob = model_sn2_13pt.predict(X_sn2_13pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn2_13pt_pred = (nn_sn2_13pt_pred_prob > prob_threshold).astype(int)

nn_acc_sn1_13pt = accuracy_score(y_sn1_13pt, nn_sn1_13pt_pred)
nn_acc_sn2_13pt = accuracy_score(y_sn2_13pt, nn_sn2_13pt_pred)

## 15 pt

In [ ]:
test_sn1_15pt_path = os.path.join(data_array_dir , "nn_test_sn1_15pt.pkl")

with open(test_sn1_15pt_path , 'rb') as f:
    X_sn1_15pt , y_sn1_15pt = pickle.load(f)

model_sn1_15pt_path = os.path.join(model_scaler_dir , "nn_sn1_15pt_model_ft_last.sav")
scaler_sn1_15pt_path = os.path.join(model_scaler_dir , "nn_sn1_15pt_scaler_ft_last.sav")

model_sn1_15pt = pickle.load(open(model_sn1_15pt_path, 'rb'))
scaler_sn1_15pt = pickle.load(open(scaler_sn1_15pt_path, 'rb'))

X_sn1_15pt_test_scaled = scaler_sn1_15pt.transform(X_sn1_15pt)
nn_sn1_15pt_pred_prob = model_sn1_15pt.predict(X_sn1_15pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn1_15pt_pred = (nn_sn1_15pt_pred_prob > prob_threshold).astype(int)

test_sn2_15pt_path = os.path.join(data_array_dir , "nn_test_sn2_15pt.pkl")

with open(test_sn2_15pt_path , 'rb') as f:
    X_sn2_15pt , y_sn2_15pt = pickle.load(f)

model_sn2_15pt_path = os.path.join(model_scaler_dir , "nn_sn2_15pt_model_ft_last.sav")
scaler_sn2_15pt_path = os.path.join(model_scaler_dir , "nn_sn2_15pt_scaler_ft_last.sav")

model_sn2_15pt = pickle.load(open(model_sn2_15pt_path, 'rb'))
scaler_sn2_15pt = pickle.load(open(scaler_sn2_15pt_path, 'rb'))

X_sn2_15pt_test_scaled = scaler_sn2_15pt.transform(X_sn2_15pt)
nn_sn2_15pt_pred_prob = model_sn2_15pt.predict(X_sn2_15pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn2_15pt_pred = (nn_sn2_15pt_pred_prob > prob_threshold).astype(int)

nn_acc_sn1_15pt = accuracy_score(y_sn1_15pt, nn_sn1_15pt_pred)
nn_acc_sn2_15pt = accuracy_score(y_sn2_15pt, nn_sn2_15pt_pred)

## 15 pt

In [ ]:
test_sn1_15pt_path = os.path.join(data_array_dir , "nn_test_sn1_15pt.pkl")

with open(test_sn1_15pt_path , 'rb') as f:
    X_sn1_15pt , y_sn1_15pt = pickle.load(f)

model_sn1_15pt_path = os.path.join(model_scaler_dir , "nn_sn1_15pt_model_ft_last.sav")
scaler_sn1_15pt_path = os.path.join(model_scaler_dir , "nn_sn1_15pt_scaler_ft_last.sav")

model_sn1_15pt = pickle.load(open(model_sn1_15pt_path, 'rb'))
scaler_sn1_15pt = pickle.load(open(scaler_sn1_15pt_path, 'rb'))

X_sn1_15pt_test_scaled = scaler_sn1_15pt.transform(X_sn1_15pt)
nn_sn1_15pt_pred_prob = model_sn1_15pt.predict(X_sn1_15pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn1_15pt_pred = (nn_sn1_15pt_pred_prob > prob_threshold).astype(int)

test_sn2_15pt_path = os.path.join(data_array_dir , "nn_test_sn2_15pt.pkl")

with open(test_sn2_15pt_path , 'rb') as f:
    X_sn2_15pt , y_sn2_15pt = pickle.load(f)

model_sn2_15pt_path = os.path.join(model_scaler_dir , "nn_sn2_15pt_model_ft_last.sav")
scaler_sn2_15pt_path = os.path.join(model_scaler_dir , "nn_sn2_15pt_scaler_ft_last.sav")

model_sn2_15pt = pickle.load(open(model_sn2_15pt_path, 'rb'))
scaler_sn2_15pt = pickle.load(open(scaler_sn2_15pt_path, 'rb'))

X_sn2_15pt_test_scaled = scaler_sn2_15pt.transform(X_sn2_15pt)
nn_sn2_15pt_pred_prob = model_sn2_15pt.predict(X_sn2_15pt_test_scaled, batch_size=4096, verbose=0).reshape(-1)
nn_sn2_15pt_pred = (nn_sn2_15pt_pred_prob > prob_threshold).astype(int)

nn_acc_sn1_15pt = accuracy_score(y_sn1_15pt, nn_sn1_15pt_pred)
nn_acc_sn2_15pt = accuracy_score(y_sn2_15pt, nn_sn2_15pt_pred)

## Accuracy + F1

In [ ]:
f1_sn1_raw = f1_score(y_sn1_raw, nn_sn1_raw_pred)
f1_sn1_3pt = f1_score(y_sn1_3pt, nn_sn1_3pt_pred)
f1_sn1_5pt = f1_score(y_sn1_5pt, nn_sn1_5pt_pred)
f1_sn1_5pt = f1_score(y_sn1_5pt, nn_sn1_5pt_pred)
f1_sn1_9pt = f1_score(y_sn1_9pt, nn_sn1_9pt_pred)
f1_sn1_11pt = f1_score(y_sn1_11pt, nn_sn1_11pt_pred)
f1_sn1_13pt = f1_score(y_sn1_13pt, nn_sn1_13pt_pred)
f1_sn1_15pt = f1_score(y_sn1_15pt, nn_sn1_15pt_pred)
f1_sn1_15pt = f1_score(y_sn1_15pt, nn_sn1_15pt_pred)

f1_sn2_raw = f1_score(y_sn2_raw, nn_sn2_raw_pred)
f1_sn2_3pt = f1_score(y_sn2_3pt, nn_sn2_3pt_pred)
f1_sn2_5pt = f1_score(y_sn2_5pt, nn_sn2_5pt_pred)
f1_sn2_5pt = f1_score(y_sn2_5pt, nn_sn2_5pt_pred)
f1_sn2_9pt = f1_score(y_sn2_9pt, nn_sn2_9pt_pred)
f1_sn2_11pt = f1_score(y_sn2_11pt, nn_sn2_11pt_pred)
f1_sn2_13pt = f1_score(y_sn2_13pt, nn_sn2_13pt_pred)
f1_sn2_15pt = f1_score(y_sn2_15pt, nn_sn2_15pt_pred)
f1_sn2_15pt = f1_score(y_sn2_15pt, nn_sn2_15pt_pred)

In [ ]:
run_points = [1 , 3 , 5 , 7  , 9 , 11 , 13 , 15 , 17]



f1_sn1 = [f1_sn1_raw , f1_sn1_3pt , f1_sn1_5pt , f1_sn1_5pt , f1_sn1_9pt , f1_sn1_11pt , f1_sn1_13pt ,
         f1_sn1_15pt , f1_sn1_15pt]

f1_sn2 = [f1_sn2_raw , f1_sn2_3pt , f1_sn2_5pt , f1_sn2_5pt , f1_sn2_9pt , f1_sn2_11pt , f1_sn2_13pt ,
         f1_sn2_15pt , f1_sn2_15pt]



accuracies_sn1 = [nn_acc_sn1_raw , nn_acc_sn1_3pt , nn_acc_sn1_5pt, nn_acc_sn1_5pt,
                 nn_acc_sn1_9pt , nn_acc_sn1_11pt , nn_acc_sn1_13pt , nn_acc_sn1_15pt,
                 nn_acc_sn1_15pt]

accuracies_sn2 = [nn_acc_sn2_raw , nn_acc_sn2_3pt , nn_acc_sn2_5pt, nn_acc_sn2_5pt,
                 nn_acc_sn2_9pt , nn_acc_sn2_11pt , nn_acc_sn2_13pt , nn_acc_sn2_15pt,
                 nn_acc_sn2_15pt]

plt.figure(figsize=[12,8])
plt.plot(run_points ,accuracies_sn1 , label = "sn1" , marker = "o" , color = "green" )
plt.plot(run_points ,accuracies_sn2 , label = "sn2" , marker = "o" , color = "red")
plt.plot(run_points ,f1_sn1 , label = "sn1" , marker = "x" , color = "green" )
plt.plot(run_points ,f1_sn2 , label = "sn2" , marker = "x" , color = "red")

plt.ylim(0.670 , 1)
plt.xticks(fontsize = 14)
plt.yticks(fontsize = 14)
plt.xlabel("Window size / n" , fontsize= 16)
plt.ylabel("Accuracy" , fontsize = 16)
plt.legend()
plt.show()

## INDICATE ALL ACCURACIES AND F1'S HERE

In [ ]:
print(f"Accuracy sn1: {accuracies_sn1}\n\nAccuracy sn2: {accuracies_sn2}\n\nF1 sn1: {f1_sn1}\n\nF1 sn2: {f1_sn2}")

# CALIBRATION ERROR(WITH n = 10 BINS)

In [ ]:
n_bins = 10

In [ ]:
ece_raw_sn1 = ece(y_sn1_raw ,nn_sn1_raw_pred_prob , n_bins = n_bins )
ece_raw_sn2 = ece(y_sn2_raw ,nn_sn2_raw_pred_prob , n_bins = n_bins )

ece_3pt_sn1 = ece(y_sn1_3pt ,nn_sn1_3pt_pred_prob , n_bins = n_bins )
ece_3pt_sn2 = ece(y_sn2_3pt ,nn_sn2_3pt_pred_prob , n_bins = n_bins )

ece_5pt_sn1 = ece(y_sn1_5pt ,nn_sn1_5pt_pred_prob , n_bins = n_bins )
ece_5pt_sn2 = ece(y_sn2_5pt ,nn_sn2_5pt_pred_prob , n_bins = n_bins )

ece_5pt_sn1 = ece(y_sn1_5pt ,nn_sn1_5pt_pred_prob , n_bins = n_bins )
ece_5pt_sn2 = ece(y_sn2_5pt ,nn_sn2_5pt_pred_prob , n_bins = n_bins )

ece_9pt_sn1 = ece(y_sn1_9pt ,nn_sn1_9pt_pred_prob , n_bins = n_bins )
ece_9pt_sn2 = ece(y_sn2_9pt ,nn_sn2_9pt_pred_prob , n_bins = n_bins )

ece_11pt_sn1 = ece(y_sn1_11pt ,nn_sn1_11pt_pred_prob , n_bins = n_bins )
ece_11pt_sn2 = ece(y_sn2_11pt ,nn_sn2_11pt_pred_prob , n_bins = n_bins )

ece_13pt_sn1 = ece(y_sn1_13pt ,nn_sn1_13pt_pred_prob , n_bins = n_bins )
ece_13pt_sn2 = ece(y_sn2_13pt ,nn_sn2_13pt_pred_prob , n_bins = n_bins )

ece_15pt_sn1 = ece(y_sn1_15pt ,nn_sn1_15pt_pred_prob , n_bins = n_bins )
ece_15pt_sn2 = ece(y_sn2_15pt ,nn_sn2_15pt_pred_prob , n_bins = n_bins )

ece_15pt_sn1 = ece(y_sn1_15pt ,nn_sn1_15pt_pred_prob , n_bins = n_bins )
ece_15pt_sn2 = ece(y_sn2_15pt ,nn_sn2_15pt_pred_prob , n_bins = n_bins )

In [ ]:
run_points = [1 , 3 , 5 , 7  , 9 , 11 , 13 , 15 , 17]

ece_sn1 = [ece_raw_sn1 , ece_3pt_sn1 , ece_5pt_sn1, ece_5pt_sn1,
                 ece_9pt_sn1 , ece_11pt_sn1 , ece_13pt_sn1 , ece_15pt_sn1,
                 ece_15pt_sn1]

ece_sn2 = [ece_raw_sn2 , ece_3pt_sn2 , ece_5pt_sn2, ece_5pt_sn2,
                 ece_9pt_sn2 , ece_11pt_sn2 , ece_13pt_sn2 , ece_15pt_sn2,
                 ece_15pt_sn2]

av_ece = 0.5*(np.array(ece_sn1) + np.array(ece_sn2))

plt.figure(figsize=[12,8])
plt.plot(run_points ,ece_sn1 , label = "sn1" , linestyle = "dotted" , marker = "o" , color = "green")
plt.plot(run_points ,ece_sn2 , label = "sn2", linestyle = "dotted" , marker = "o" , color = "red" )
plt.plot(run_points ,av_ece , label = "averaged", linestyle = "dashed" , marker = "o" , color = "purple" )
plt.xticks([1 , 3  , 5 , 7 , 9 , 11 , 13 , 15 , 17],fontsize = 18)
plt.yticks(fontsize = 18)
#plt.ylim(0.670 , 1)
plt.xlim(-1,18)
plt.xlabel("Window size / n" , fontsize = 18)
plt.ylabel("ECE" , fontsize = 18)
plt.legend(fontsize = 16)
plt.title("Expected Calibration Error -- NN" , fontsize = 20)
plt.show()

## SELECTION CRITERION:

\begin{equation}
    n^{*} = \underset{n}{\mathrm{argmin}} \left( \frac{|ECE_{\mathrm{sn1}}(n) - ECE_{\mathrm{sn2}}(n)|}{ACC_{\mathrm{sn1}}(n) + ACC_{\mathrm{sn2}}(n)} \right)
\end{equation}

In [ ]:
###implemented here. pick the window corresponding to smallest number.

n_stars = np.abs(np.array(ece_sn1) - np.array(ece_sn2))/(np.array(accuracies_sn1) + np.array(accuracies_sn2))
n_stars